<a href="https://colab.research.google.com/github/Das-Debjit/Harmful-Language-Detection/blob/main/notebooks/Harmful_Language_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# This command lists all files and folders inside your project directory
!ls "/content/drive/My Drive/Harmful Language Detection/"

In [ ]:
# This command lists all files and folders inside your project directory
!ls "/content/drive/My Drive/Harmful Language Detection/jigsaw toxic comment classification"

## Setup and Library Imports

In [ ]:
# Import libraries for data manipulation and analysis
import pandas as pd
import numpy as np
import joblib

# Import libraries for text cleaning and processing
import re

# Import libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# To display plots (optional but good practice)
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully!")

## Load the Datasets from Google Drive

In [ ]:
# Define the correct paths to your files
hate_speech_path = '/content/drive/My Drive/Harmful Language Detection/Hate Speech and Offensive Language Dataset.csv'
jigsaw_path = '/content/drive/My Drive/Harmful Language Detection/jigsaw toxic comment classification/train.csv'

# Load the Hate Speech and Offensive Language dataset
try:
    hate_speech_df = pd.read_csv(hate_speech_path)
    print("--- Hate Speech Dataset ---")
    print(f"Successfully loaded from: {hate_speech_path}")
    print(f"Shape: {hate_speech_df.shape}")
    print(hate_speech_df.head())
    print("\n")
except FileNotFoundError:
    print(f"Error: File not found at '{hate_speech_path}'. Please double-check the path.")

# Load the Jigsaw Toxic Comment dataset
try:
    jigsaw_df = pd.read_csv(jigsaw_path)
    print("--- Jigsaw Toxic Comment Dataset ---")
    print(f"Successfully loaded from: {jigsaw_path}")
    print(f"Shape: {jigsaw_df.shape}")
    print(jigsaw_df.head())
except FileNotFoundError:
    print(f"Error: File not found at '{jigsaw_path}'. Please double-check the path.")

## Preprocess the Hate Speech Dataset

In [ ]:
print("--- Preprocessing Hate Speech Dataset ---")

# Combine 'hate_speech' (0) and 'offensive_language' (1) into a single 'toxic' category (1)
# 'neither' (2) will be 'not_toxic' (0).
hate_speech_df['label'] = hate_speech_df['class'].apply(lambda x: 1 if x < 2 else 0)
hate_speech_df.rename(columns={'tweet': 'text'}, inplace=True)

# Create a new, clean dataframe
clean_hate_speech_df = hate_speech_df[['text', 'label']]

print("\nNew label distribution (1=Toxic, 0=Not Toxic):")
print(clean_hate_speech_df['label'].value_counts())

## Preprocess the Jigsaw Dataset

In [ ]:
print("--- Preprocessing Jigsaw Dataset ---")

# If any of the toxic columns are 1, we'll mark the comment as toxic (1).
toxic_labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
jigsaw_df['label'] = jigsaw_df[toxic_labels].max(axis=1)
jigsaw_df.rename(columns={'comment_text': 'text'}, inplace=True)

# Create a new, clean dataframe
clean_jigsaw_df = jigsaw_df[['text', 'label']]

print("\nNew label distribution (1=Toxic, 0=Not Toxic):")
print(clean_jigsaw_df['label'].value_counts())

## Combine & Shuffle Datasets

In [ ]:
# Combine the two dataframes
combined_df = pd.concat([clean_hate_speech_df, clean_jigsaw_df], ignore_index=True)

# Shuffle the dataset
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- Combined Dataset ---")
print(f"Total shape: {combined_df.shape}")
print("\nFinal combined DataFrame head:")
print(combined_df.head())

## Text Cleaning Function

In [ ]:
def clean_text(text):
    text = text.lower()  # Lowercase text
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    return text

print("Cleaning text data... this may take a moment.")
combined_df['cleaned_text'] = combined_df['text'].apply(clean_text)
print("✅ Text cleaning complete.")

print("\n--- Example of Cleaning ---")
print("Original: ", combined_df['text'].iloc[5])
print("Cleaned:  ", combined_df['cleaned_text'].iloc[5])

## Create Checkpoint (Save Cleaned Data)

In [ ]:
# Define a path to save your cleaned data
cleaned_data_path = '/content/drive/My Drive/Harmful Language Detection/cleaned_toxic_comments.csv'

# Save the DataFrame to a CSV file
print(f"Saving cleaned data to: {cleaned_data_path}")
combined_df.to_csv(cleaned_data_path, index=False)

print("✅ Checkpoint created successfully!")

## Split Data and Build Model Pipeline

In [ ]:
# Define features (X) and target (y)
X = combined_df['cleaned_text'].astype(str) # Ensure all data is string type
y = combined_df['label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Data Splitting ---")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Create a model pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(solver='liblinear', random_state=42))
])

print("\n✅ Model pipeline created successfully.")

## Train the Model

In [ ]:
print("--- Model Training ---")
print("Training the model... please wait.")

# Train the model
model.fit(X_train, y_train)

print("✅ Model training complete!")

## Evaluate the Model

In [ ]:
print("--- Model Evaluation ---")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Toxic', 'Toxic']))

# Print the confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Toxic', 'Toxic'], yticklabels=['Not Toxic', 'Toxic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

## Test with New Sentences

In [ ]:
# Test the model with some new phrases
test_sentences = [
    "I love this, thank you so much for your help!",
    "You are an idiot and I am going to find you.",
    "This is the worst thing I have ever seen.",
    "I don't agree with your opinion on this matter."
]

# Predict the sentiment of the new sentences
predictions = model.predict(test_sentences)
prediction_proba = model.predict_proba(test_sentences)

print("--- Testing with New Sentences ---")
for sentence, pred, proba in zip(test_sentences, predictions, prediction_proba):
    label = "Toxic" if pred == 1 else "Not Toxic"
    print(f"\nSentence: '{sentence}'")
    print(f"Predicted Label: {label}")
    print(f"Confidence -> Not Toxic: {proba[0]:.2f} | Toxic: {proba[1]:.2f}")

## Save the Baseline Model

In [ ]:
import joblib

# Define the path for saving the model
model_save_path = '/content/drive/My Drive/Harmful Language Detection/toxic_comment_baseline_model.joblib'

# Save the model
print(f"Saving model to: {model_save_path}")
joblib.dump(model, model_save_path)

print("✅ Baseline model saved successfully!")

## Load and Test the Saved Model

In [ ]:
# Load the model from the file
loaded_model = joblib.load(model_save_path)

print("✅ Model loaded successfully!")

# Test the loaded model with a new sentence
test_sentence = ["you are a wonderful person and so helpful"]
prediction = loaded_model.predict(test_sentence)
prediction_proba = loaded_model.predict_proba(test_sentence)

label = "Toxic" if prediction[0] == 1 else "Not Toxic"
print(f"\nTest Sentence: '{test_sentence[0]}'")
print(f"Predicted Label: {label}")
print(f"Confidence -> Not Toxic: {prediction_proba[0][0]:.2f} | Toxic: {prediction_proba[0][1]:.2f}")

## A More Advanced Scraping Function

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def scrape_script_from_imsdb_v3(url):
    """
    Scrapes dialogue from imsdb.com using a hybrid approach.
    It first tries to parse by HTML <b> tags, and if that fails,
    it falls back to a regex-based text parsing method.
    """
    print(f"Scraping: {url}")
    dialogues = []

    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        script_content = soup.find('pre')
        if not script_content:
            print(f" -> Could not find script content block.")
            return pd.DataFrame()

        # --- METHOD 1: Parse using <b> tags (more reliable if available) ---
        current_character = None
        for element in script_content.children:
            if element.name == 'b':
                current_character = element.get_text(strip=True)
            elif isinstance(element, str) and element.strip() and current_character:
                dialogue_line = element.strip()
                # Basic filter to avoid script directions
                if not dialogue_line.startswith('(') and not dialogue_line.endswith(')'):
                    dialogues.append({'character': current_character, 'dialogue': dialogue_line})

        # --- METHOD 2: Fallback to Regex if Method 1 failed ---
        if len(dialogues) < 20: # If we found very few dialogues, try the other method
            print(" -> <b> tag method found few results, trying regex fallback...")
            dialogues = [] # Reset the list

            lines = script_content.get_text().split('\n')
            character = ""
            character_pattern = re.compile(r'^\s{20,}([A-Z][A-Z\s\.]+)$') # Character name is all caps, heavily indented
            dialogue_pattern = re.compile(r'^\s{10,}[a-zA-Z]') # Dialogue is less indented

            for line in lines:
                char_match = character_pattern.match(line)
                dialogue_match = dialogue_pattern.match(line)

                if char_match:
                    character = char_match.group(1).strip()
                elif dialogue_match and character:
                    dialogue_line = line.strip()
                    dialogues.append({'character': character, 'dialogue': dialogue_line})

        print(f" -> Found {len(dialogues)} lines of dialogue.")
        return pd.DataFrame(dialogues)

    except Exception as e:
        print(f" -> Failed to scrape {url}. Error: {e}")
        return pd.DataFrame()

print("✅ Hybrid scraping function (v3) created.")

## Scrape Multiple Scripts and Combine

In [ ]:
# List of new movie scripts to scrape from imsdb.com
urls_to_scrape = [
    'https://imsdb.com/scripts/Godfather.html',
    'https://imsdb.com/scripts/Reservoir-Dogs.html',
    'https://imsdb.com/scripts/Pulp-Fiction.html' # Let's re-scrape Pulp Fiction with the better function
]

# Path to your combined dataset
combined_dialogue_path = '/content/drive/My Drive/Harmful Language Detection/combined_crime_dialogues.csv'

# Start with an empty DataFrame
all_dialogues_df = pd.DataFrame()
print("Starting with a fresh dataset.")

# Scrape each new URL and add the results to our main DataFrame
for url in urls_to_scrape:
    new_dialogues_df = scrape_script_from_imsdb(url)
    if not new_dialogues_df.empty:
        # Add a column to know which movie the dialogue is from
        movie_title = url.split('/')[-1].replace('.html', '').replace('-', ' ').title()
        new_dialogues_df['movie'] = movie_title

        all_dialogues_df = pd.concat([all_dialogues_df, new_dialogues_df], ignore_index=True)

    time.sleep(1) # Wait 1 second between requests

# Save the final, combined DataFrame
if not all_dialogues_df.empty:
    print("\n--- Saving Combined Dataset ---")
    print(f"Total dialogues from all movies: {len(all_dialogues_df)}")
    all_dialogues_df.to_csv(combined_dialogue_path, index=False)
    print(f"✅ All dialogues saved to: {combined_dialogue_path}")

    print("\n--- Sample from the final dataset ---")
    print(all_dialogues_df.sample(10))
else:
    print("\nNo dialogues were scraped. The final file was not saved.")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def scrape_script_from_imsdb_v3(url):
    """
    Scrapes dialogue from imsdb.com using a hybrid approach.
    It first tries to parse by HTML <b> tags, and if that fails,
    it falls back to a regex-based text parsing method.
    """
    print(f"Scraping: {url}")
    dialogues = []

    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        script_content = soup.find('pre')
        if not script_content:
            print(f" -> Could not find script content block.")
            return pd.DataFrame()

        # --- METHOD 1: Parse using <b> tags ---
        current_character = None
        for element in script_content.children:
            if element.name == 'b':
                current_character = element.get_text(strip=True)
            elif isinstance(element, str) and element.strip() and current_character:
                dialogue_line = element.strip()
                if not dialogue_line.startswith('(') and not dialogue_line.endswith(')'):
                    dialogues.append({'character': current_character, 'dialogue': dialogue_line})

        # --- METHOD 2: Fallback to Regex if Method 1 failed ---
        if len(dialogues) < 20:
            print(" -> <b> tag method found few results, trying regex fallback...")
            dialogues = [] # Reset the list

            lines = script_content.get_text().split('\n')
            character = ""
            character_pattern = re.compile(r'^\s{20,}([A-Z][A-Z\s\.]+)$')
            dialogue_pattern = re.compile(r'^\s{10,}[a-zA-Z]')

            for line in lines:
                char_match = character_pattern.match(line)
                dialogue_match = dialogue_pattern.match(line)
                if char_match:
                    character = char_match.group(1).strip()
                elif dialogue_match and character:
                    dialogue_line = line.strip()
                    dialogues.append({'character': character, 'dialogue': dialogue_line})

        print(f" -> Found {len(dialogues)} lines of dialogue.")
        return pd.DataFrame(dialogues)

    except Exception as e:
        print(f" -> Failed to scrape {url}. Error: {e}")
        return pd.DataFrame()

# --- Main Execution ---

# 1. Define the new list of URLs you provided
urls_to_scrape = [
    'https://imsdb.com/scripts/American-Gangster.html',
    'https://imsdb.com/scripts/John-Wick-Chapter-4.html',
    'https://imsdb.com/scripts/Joker.html',
    'https://imsdb.com/scripts/Sherlock-Holmes.html',
    'https://imsdb.com/scripts/Catch-Me-If-You-Can.html',
    'https://imsdb.com/scripts/Saw.html',
    'https://imsdb.com/scripts/Dark-Knight-Rises,-The.html'
]

# 2. Define the path to your dataset
combined_dialogue_path = '/content/drive/My Drive/Harmful Language Detection/combined_crime_dialogues.csv'

# 3. Load your existing data
try:
    all_dialogues_df = pd.read_csv(combined_dialogue_path)
    print(f"Loaded existing dataset with {len(all_dialogues_df)} dialogues.")
except FileNotFoundError:
    all_dialogues_df = pd.DataFrame()
    print("No existing dialogue file found. Starting a new one.")

# 4. Scrape each new URL and append the results
for url in urls_to_scrape:
    new_dialogues_df = scrape_script_from_imsdb_v3(url)
    if not new_dialogues_df.empty:
        movie_title = url.split('/')[-1].replace('.html', '').replace('-', ' ').title()
        new_dialogues_df['movie'] = movie_title
        all_dialogues_df = pd.concat([all_dialogues_df, new_dialogues_df], ignore_index=True)
    time.sleep(1) # Wait 1 second between requests

# 5. Save the final, updated DataFrame
if not all_dialogues_df.empty:
    print("\n--- Saving Final Combined Dataset ---")
    # Remove duplicate dialogues just in case we scraped the same movie twice
    all_dialogues_df.drop_duplicates(subset=['dialogue', 'movie'], inplace=True)
    print(f"Total dialogues from all movies: {len(all_dialogues_df)}")
    all_dialogues_df.to_csv(combined_dialogue_path, index=False)
    print(f"✅ All dialogues saved to: {combined_dialogue_path}")

    print("\n--- Sample from the final dataset ---")
    print(all_dialogues_df.sample(10))
else:
    print("\nNo new dialogues were scraped. The final file was not saved.")

## Part 2: High-End Model (DistilBERT) for Intent Classification
 We will use our new custom dataset to train a powerful transformer-based model (DistilBERT) to classify dialogue into 5 specific intents.

In [ ]:

!pip install transformers[torch] datasets evaluate

### Generate Large Randomized Synthetic Dataset

In [ ]:
# Cell (New): Generate Large Randomized Synthetic Dataset (Approx 6000+ Rows)
import pandas as pd
import random

# --- Configuration ---
MIN_SAMPLES_PER_CATEGORY = 1000
MAX_SAMPLES_PER_CATEGORY = 1400
PROJECT_FOLDER = "/content/drive/My Drive/Harmful Language Detection/"
OUTPUT_FILENAME_RANDOM = PROJECT_FOLDER + "synthetic_intent_dataset_6k_randomized.csv" # New filename

# --- Templates and Fillers (Copied from previous cell - ensure they are defined) ---

# Neutrals (Business, Casual, Observations)
neutral_templates = [
    "The report indicates a {} trend in sales.", "Please forward the meeting minutes from {}.",
    "Let's schedule a {} call for next week.", "Did you see the news about {}?",
    "I'm planning to {} this weekend.", "The weather seems {} today.",
    "Remember to submit your {} by Friday.", "The system update is scheduled for {}.",
    "Client feedback on {} was generally positive.", "We need to analyze the {} data more carefully.",
    "The presentation on {} was very insightful.", "Can you help me with {}?",
    "I just finished reading about {}.", "The traffic was {} this morning.",
    "Let's grab {} later.", "How was your {}?",
    "The {} project deadline is approaching.", "We should discuss the {} strategy.",
    "The office needs more {} supplies.", "Training session for {} is mandatory."
]
neutral_fillers = {
    "trend": ["positive", "negative", "stable", "upward", "downward"],
    "meeting_topic": ["yesterday's meeting", "the budget review", "the Q3 planning session"],
    "call_type": ["quick sync", "detailed discussion", "follow-up"],
    "news_subject": ["the election", "the market fluctuations", "the recent tech advancements"],
    "weekend_activity": ["relax", "visit family", "catch up on work", "go hiking"],
    "weather_desc": ["nice", "gloomy", "unpredictable", "stormy"],
    "document_type": ["expense report", "project proposal", "timesheet"],
    "time_desc": ["tonight", "tomorrow morning", "Sunday night"],
    "project_name": ["Project Alpha", "the Delta initiative", "the marketing campaign"],
    "data_type": ["sales", "user engagement", "competitor"],
    "topic": ["AI ethics", "market trends", "the new software"],
    "task": ["this spreadsheet", "the presentation slides", "debugging this code"],
    "subject": ["the new policy", "that article", "the historical event"],
    "traffic_condition": ["light", "heavy", "terrible"],
    "food_drink": ["coffee", "lunch", "a drink"],
    "event": ["vacation", "conference", "presentation"],
    "item": ["printer paper", "coffee", "cleaning"]
}

# Confessions (Admissions of guilt, regret, mistakes - varied severity)
confession_templates = [
    "Alright, I admit it, I {}.", "It was me who {}.",
    "I have to confess, I {}.", "Yes, I'm the one responsible for {}.",
    "I shouldn't have {}, I regret it now.", "I made a mistake when I {}.",
    "I took {} without asking.", "I lied about {} earlier.",
    "I was involved in {}.", "Honestly, I {} and I feel terrible.",
    "I broke {} and didn't tell anyone.", "The truth is, I {}.",
    "I take full responsibility for {}.", "I {} because I thought I had to.",
    "Looking back, I wish I hadn't {}.", "I blame myself for {}."
]
confession_fillers = {
    "action_past": [
        "took the money", "made the call", "deleted the files", "spread the rumor",
        "faked the report", "skipped the meeting", "lost the key", "broke the vase",
        "ate the last piece of cake", "used your password", "was there that night",
        "helped him escape", "ignored the warning", "manipulated the results",
        "covered it up", "placed the bet", "started the fight", "crashed the car"
    ],
    "item_taken": ["the file", "your pen", "the evidence", "the credit card"],
    "thing_lied_about": ["my alibi", "where I was", "the results", "my qualifications"],
    "event_involved": ["the robbery", "the cover-up", "the whole scheme", "that incident"],
    "thing_broken": ["the window", "the rules", "my promise", "the equipment"]
}


# Planning Crime (Coordination, steps, logistics)
planning_templates = [
    "First, we {}. Then, you {}.", "The plan is to {} at {}.",
    "We need {} before we can proceed.", "Make sure {} is ready by {}.",
    "The target is {}. We move on {}.", "Coordinate with {} for the next phase.",
    "The {} needs to be disabled.", "Use {} for communication.",
    "The rendezvous point is {}.", "Acquire {} by tomorrow.",
    "We need to create {} as a diversion.", "The escape route involves {}.",
    "Plant {} in the location.", "The signal will be {}.",
    "Secure {} before initiating.", "The objective is to {}.",
    "We need to establish {} beforehand.", "The {} phase begins at midnight."
]
planning_fillers = {
    "action_step1": ["disable the cameras", "secure the perimeter", "cut the power", "create a diversion"],
    "action_step2": ["pick the lock", "grab the files", "make the switch", "plant the device"],
    "location_target": ["the bank", "the warehouse", "the museum", "the convoy"],
    "time_specific": ["dawn", "midnight", "3 AM", "noon"],
    "resource_needed": ["the access codes", "burner phones", "a getaway car", "inside information"],
    "item_ready": ["the van", "the equipment", "the forged documents"],
    "target_desc": ["vulnerable during transit", "alone at night", "expecting a delivery"],
    "signal_time": ["my signal", "the third bell", "the lights go out"],
    "contact_person": ["the driver", "our inside man", "the hacker"],
    "system_disable": ["alarm system", "security cameras", "tracking device"],
    "tool_communication": ["encrypted channels", "coded messages", "burner phones"],
    "location_meet": ["the old mill", "Pier 12", "the abandoned airfield"],
    "item_acquire": ["the blueprints", "unmarked weapons", "the master keycard"],
    "diversion_type": ["a fire alarm", "a staged accident", "a loud noise"],
    "escape_method": ["the service tunnels", "the subway system", "the river"],
    "item_plant": ["the listening device", "the evidence", "the bomb"],
    "signal_desc": ["three knocks", "a flashing light", "the codeword 'Phoenix'"],
    "asset_secure": ["the perimeter", "the exits", "the hostage"],
    "objective": ["retrieve the ledger", "steal the diamonds", "extract the target"],
    "prep_needed": ["an alibi", "a safe house", "false IDs"],
    "operation_phase": ["infiltration", "hacking", "extraction"]
}

# Threatening Action (Direct, veiled, conditional)
threatening_templates = [
    "If you {}, then {}.", "You better {}, or else {}.",
    "Don't make me {}.", "This is your last chance to {}.",
    "People who {} tend to {}.", "I know {}. Don't test me.",
    "Think carefully before you {}.", "I suggest you {}.",
    "Cross me again, and {}.", "Failure will result in {}.",
    "Your {} depends on your cooperation.", "I have ways of dealing with {}.",
    "Don't underestimate {}.", "This conversation {}.",
    "Comply, or face {}.", "I can make {} very difficult.",
    "One wrong move could lead to {}.", "Keep {} out of this."
]
threatening_fillers = {
    "condition_action": ["talk to the cops", "cross me", "fail me", "interfere", "betray me"],
    "consequence_neg": ["your family pays", "you disappear", "you'll regret it", "I end you"],
    "action_forced": ["come over there", "escalate this", "do something drastic", "get involved"],
    "action_avoid_neg": ["back off", "walk away", "keep quiet", "comply"],
    "group_behavior": ["snitches", "people who owe me", "those who betray us"],
    "outcome_neg": ["end up in ditches", "disappear", "have accidents", "pay dearly"],
    "sensitive_info": ["where you live", "your secret", "where your kids go"],
    "action_careful": ["speak", "act", "make your next move"],
    "suggestion_veiled": ["forget you saw me", "reconsider your position", "walk away now"],
    "consequence_repeat": ["you won't walk away", "it won't be pleasant", "I won't be so forgiving"],
    "result_failure": ["severe consequences", "pain", "unpleasantness"],
    "wellbeing_type": ["safety", "future", "family's well-being"],
    "problem_type": ["people like you", "obstacles", "loose ends"],
    "power_source": ["my reach", "my connections", "what I'm capable of"],
    "conversation_status": ["never happened", "stays between us", "ends now"],
    "face_consequence": ["the pain", "my wrath", "unimaginable trouble"],
    "difficulty_area": ["your life", "your business", "things for you"],
    "result_mistake": ["unforeseen accidents", "a bad outcome", "serious trouble"],
    "person_object_safe": ["your family", "my name", "this business"]
}


# Discussing Illegal Activity (Past events, general knowledge, reporting)
discussing_templates = [
    "Did you hear about {}?", "They were talking about {}.",
    "Remember when {}?", "Apparently, {} happened last night.",
    "He got busted for {}.", "The news reported on {}.",
    "She used to be involved in {}.", "That whole {} operation was messy.",
    "I heard {} got away with it.", "They found evidence related to {}.",
    "The investigation into {} is ongoing.", "He spent time in prison for {}.",
    "It was all over the papers about {}.", "The authorities cracked down on {}.",
    "There's a rumor going around about {}.", "He bragged about {}."
]
discussing_fillers = {
    "event_crime": [
        "the robbery downtown", "the big heist", "that drug bust", "the art theft",
        "the police raid", "the shootout", "the prison escape", "the smuggling operation",
        "the counterfeit ring", "the data breach", "the insider trading scandal"
    ],
    "person_involved": ["the boss", "that crew", "the informant", "the getaway driver"],
    "past_action": [
        "they pulled off that bank job", "we almost got caught", "he snitched on his partners"
    ],
    "illegal_activity": [
        "money laundering", "extortion", "arms dealing", "running numbers",
        "fixing races", "bribery", "drug trafficking", "identity theft"
    ]
}

# --- Generation Function (same as before) ---
def generate_sentence(template, fillers):
    sentence = template
    # Fill specific placeholders first
    for key, options in fillers.items():
        placeholder = "{" + key + "}"
        if placeholder in sentence:
            sentence = sentence.replace(placeholder, random.choice(options), 1)
    # Fill generic placeholders if any remain
    count_generic = sentence.count("{}")
    if count_generic > 0:
        available_keys = list(fillers.keys())
        for _ in range(count_generic):
            if available_keys:
                key_to_use = random.choice(available_keys)
                sentence = sentence.replace("{}", random.choice(fillers[key_to_use]), 1)
            else:
                key_to_use = random.choice(list(fillers.keys()))
                sentence = sentence.replace("{}", random.choice(fillers[key_to_use]), 1)
    return sentence

# --- Populate the lists with RANDOMIZED counts ---
all_dialogues = []
all_intents = []
category_counts = {} # To store the random counts

categories = {
    "neutral": (neutral_templates, neutral_fillers),
    "confession": (confession_templates, confession_fillers),
    "planning_crime": (planning_templates, planning_fillers),
    "threatening_action": (threatening_templates, threatening_fillers),
    "discussing_illegal_activity": (discussing_templates, discussing_fillers)
}

print("Generating randomized synthetic data...")
for intent, (templates, fillers) in categories.items():
    # **Generate a random number of samples for this category**
    num_samples_this_category = random.randint(MIN_SAMPLES_PER_CATEGORY, MAX_SAMPLES_PER_CATEGORY)
    category_counts[intent] = num_samples_this_category
    print(f"Targeting {num_samples_this_category} samples for '{intent}'...")

    count = 0
    while count < num_samples_this_category:
        template = random.choice(templates)
        sentence = generate_sentence(template, fillers)
        if len(sentence.split()) > 3: # Basic quality check
            all_dialogues.append(sentence)
            all_intents.append(intent)
            count += 1
    print(f"--> Generated {count} samples.")

# --- Create DataFrame ---
total_generated = len(all_dialogues)
synthetic_df_random = pd.DataFrame({
    'character': ['SYNTHETIC_GEN_RND'] * total_generated,
    'dialogue': all_dialogues,
    'movie': ['SYNTHETIC_DATASET_RND'] * total_generated,
    'intent_category': all_intents
})

# Shuffle the dataset
synthetic_df_random = synthetic_df_random.sample(frac=1, random_state=42).reset_index(drop=True)

# --- Save the Dataset ---
try:
    synthetic_df_random.to_csv(OUTPUT_FILENAME_RANDOM, index=False)
    print("\n--- SUCCESS ---")
    print(f"Generated {len(synthetic_df_random)} total synthetic samples.")
    print(f"✅ Randomized synthetic dataset saved to: {OUTPUT_FILENAME_RANDOM}")
    print("\n--- Actual Dataset Distribution ---")
    print(synthetic_df_random['intent_category'].value_counts())
    print("\n--- Target vs Actual ---")
    print(category_counts) # Show the target counts we aimed for

except Exception as e:
    print(f"\n--- ERROR ---")
    print(f"Could not save the file. Error: {e}")

### Combine Real and Synthetic Datasets

In [ ]:
# Cell (New): Combine Real and Synthetic Datasets
import pandas as pd
import os # Import os for path checking

# --- Define File Paths ---
PROJECT_FOLDER = "/content/drive/My Drive/Harmful Language Detection/"
ORIGINAL_CUSTOM_DATA_PATH = PROJECT_FOLDER + "classified_crime_dialogues_priority_sweep_corrected.csv"
SYNTHETIC_DATA_PATH = PROJECT_FOLDER + "synthetic_intent_dataset_6k_randomized.csv"
COMBINED_OUTPUT_PATH = PROJECT_FOLDER + "combined_real_synthetic_dialogues.csv" # New output file

print("Attempting to combine datasets...")

try:
    # --- Load Original Data ---
    if not os.path.exists(ORIGINAL_CUSTOM_DATA_PATH):
        raise FileNotFoundError(f"Original dataset not found: {ORIGINAL_CUSTOM_DATA_PATH}")
    df_original = pd.read_csv(ORIGINAL_CUSTOM_DATA_PATH)
    print(f"Loaded original dataset with {len(df_original)} entries.")

    # --- Load Synthetic Data ---
    if not os.path.exists(SYNTHETIC_DATA_PATH):
        raise FileNotFoundError(f"Synthetic dataset not found: {SYNTHETIC_DATA_PATH}")
    df_synthetic = pd.read_csv(SYNTHETIC_DATA_PATH)
    print(f"Loaded synthetic dataset with {len(df_synthetic)} entries.")

    # --- Combine ---
    # Ensure columns match (or select only the ones needed: dialogue, intent_category)
    # Assuming both have 'dialogue' and 'intent_category'
    df_combined = pd.concat([
        df_original[['dialogue', 'intent_category']], # Select necessary columns
        df_synthetic[['dialogue', 'intent_category']]  # Select necessary columns
    ], ignore_index=True)

    # Add back placeholder columns if needed by later code (optional, can be removed)
    df_combined['character'] = 'COMBINED'
    df_combined['movie'] = 'COMBINED_DATA'

    print(f"Combined dataset created with {len(df_combined)} total entries.")

    # --- Shuffle ---
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
    print("Combined dataset shuffled.")

    # --- Save ---
    df_combined.to_csv(COMBINED_OUTPUT_PATH, index=False)
    print("\n--- SUCCESS ---")
    print(f"✅ Combined dataset saved to: {COMBINED_OUTPUT_PATH}")
    print("\n--- Combined Dataset Distribution ---")
    print(df_combined['intent_category'].value_counts())

except FileNotFoundError as fnf_error:
    print(f"\n--- ERROR ---")
    print(f"{fnf_error}")
    print("Please ensure both input CSV files exist in the correct location.")
except Exception as e:
    print(f"\n--- ERROR ---")
    print(f"An error occurred during combining/saving: {e}")

### Install libraries

In [ ]:
# Cell 2: Install Hugging Face Libraries
!pip install transformers[torch] datasets evaluate scikit-learn pandas matplotlib seaborn
print("✅ Libraries installed.")

### Import Libraries and Define Paths

In [ ]:
# Cell 3: Import Libraries and Define Paths
import pandas as pd
import numpy as np
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
import os # Import os for path checking

# --- Define project paths ---
PROJECT_FOLDER = "/content/drive/My Drive/Harmful Language Detection/"
# --- Use the NEW COMBINED dataset ---
CUSTOM_DATA_PATH = PROJECT_FOLDER + "combined_real_synthetic_dialogues.csv"
# --- Paths for outputs of THIS training run ---
COMBINED_MODEL_PATH = PROJECT_FOLDER + "high_end_intent_model_combined_v1/" # New model dir
COMBINED_LOGS_PATH = PROJECT_FOLDER + "logs_combined_v1/" # New logs dir
# --- Paths for label map and data splits (use new names) ---
LABEL_MAP_PATH = PROJECT_FOLDER + "intent_label_mapping_combined.json" # New map file
TRAIN_SPLIT_PATH = PROJECT_FOLDER + "intent_train_split_combined.csv" # New split file
TEST_SPLIT_PATH = PROJECT_FOLDER + "intent_test_split_combined.csv"   # New split file

print("✅ Libraries imported and paths defined.")
print(f"Using dataset: {CUSTOM_DATA_PATH}")
print(f"Model will be saved to: {COMBINED_MODEL_PATH}")

### Load Combined Dataset and Process Labels

In [ ]:
# Cell 4: Load Combined Dataset and Process Labels
try:
    # --- Load the Dataset ---
    df = pd.read_csv(CUSTOM_DATA_PATH)
    print(f"Successfully loaded dataset: {CUSTOM_DATA_PATH}")
    print(f"Shape: {df.shape}")

    # --- Create/Update Label Mappings ---
    intent_categories = df['intent_category'].unique()
    # Ensure consistent ordering if possible (optional but good practice)
    intent_categories = sorted(list(intent_categories))
    label2id = {label: i for i, label in enumerate(intent_categories)}
    id2label = {i: label for i, label in enumerate(intent_categories)}

    # Add the numeric 'label' column
    df['label'] = df['intent_category'].map(label2id)

    print("\n--- Label to ID Mapping ---")
    print(label2id)

    # --- Save the Label Mapping ---
    with open(LABEL_MAP_PATH, 'w') as f:
        json.dump({'label2id': label2id, 'id2label': id2label}, f)
    print(f"\n✅ Label mappings saved to {LABEL_MAP_PATH}")

    print("\n--- Data Head with 'label' column ---")
    print(df.head())

except FileNotFoundError:
    print(f"Error: File not found at '{CUSTOM_DATA_PATH}'. Did the previous cell run correctly?")
    df = None
except Exception as e:
    print(f"An error occurred: {e}")
    df = None

if df is None:
    raise ValueError("DataFrame 'df' was not loaded. Cannot proceed.")
else:
     print("\n✅ DataFrame 'df' is ready.")

### Split Data and Create Checkpoints

In [ ]:
# Cell 5: Split Data and Create Checkpoints
if df is not None:
    print("--- Splitting Data ---")
    train_df, test_df = train_test_split(
        df,
        test_size=0.2, # Using 20% for test, now on a larger dataset
        random_state=42,
        stratify=df['label'] # Stratify by the numeric label
    )

    print(f"Training data shape: {train_df.shape}")
    print(f"Testing data shape: {test_df.shape}")

    # --- Save Splits as Checkpoints ---
    train_df.to_csv(TRAIN_SPLIT_PATH, index=False)
    test_df.to_csv(TEST_SPLIT_PATH, index=False)
    print(f"✅ Train/Test splits saved as CSV checkpoints: \n{TRAIN_SPLIT_PATH}\n{TEST_SPLIT_PATH}")
else:
    print("Cannot split data because DataFrame 'df' was not loaded.")

### Load Data into Hugging Face Dataset Object

In [ ]:
# Cell 6: Load Data into Hugging Face Dataset Object
try:
    dataset = load_dataset('csv', data_files={
        'train': TRAIN_SPLIT_PATH,
        'test': TEST_SPLIT_PATH
    })
    print("\n--- Dataset Object Loaded ---")
    print(dataset)
    # Check a sample
    print("\n--- Sample Training Data ---")
    print(dataset['train'][0])
except Exception as e:
    print(f"Error loading data splits into Dataset object: {e}")
    dataset = None

if dataset is None:
     raise ValueError("Dataset object not created. Check previous steps.")

### Load Tokenizer and Process Data

In [ ]:
# Cell 7: Load Tokenizer and Process Data
if dataset is not None:
    model_checkpoint = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def tokenize_function(batch):
        # Ensure 'dialogue' column exists and handle potential None values
        texts = [str(text) if text is not None else "" for text in batch['dialogue']]
        return tokenizer(texts, padding="max_length", truncation=True, max_length=512)

    print("Tokenizing the dataset (this may take a moment)...")
    tokenized_datasets = dataset.map(tokenize_function, batched=True)

    # Dynamically find columns to remove (all except label and HF internals)
    cols_to_remove = [col for col in dataset['train'].column_names if col not in ['label']]
    print(f"Removing columns: {cols_to_remove}")

    tokenized_datasets = tokenized_datasets.remove_columns(cols_to_remove)
    tokenized_datasets.set_format("torch")
    print("✅ Tokenization complete and format set to torch.")
    print("\nExample of a tokenized item:")
    print(tokenized_datasets['train'][0])

else:
    print("Dataset object not found. Skipping tokenization.")
    tokenized_datasets = None

if tokenized_datasets is None:
    raise ValueError("Tokenized datasets not created successfully.")

### Load the Pre-trained Model

In [ ]:
# Cell 8: Load the Pre-trained Model
if 'label2id' in locals() and 'id2label' in locals():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    )
    print("✅ Pre-trained DistilBERT model loaded and configured.")
else:
    print("Error: label2id or id2label mappings not found.")
    model = None

if model is None:
    raise ValueError("Model not loaded successfully.")

###  Define Training Arguments (for Combined Data V1)

In [ ]:
# Cell 9: Define Training Arguments (for Combined Data V1)
training_args = TrainingArguments(
    output_dir=COMBINED_MODEL_PATH, # Use combined directory
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3, # Start with 3 epochs - adjust if needed based on loss curves
    per_device_train_batch_size=16, # Adjust based on GPU memory
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=COMBINED_LOGS_PATH, # Use combined logs directory
    logging_strategy="steps",
    logging_steps=100, # Log every 100 steps
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none"
)

print("✅ TrainingArguments defined for combined data training.")
print(f"Model checkpoints will be saved to: {training_args.output_dir}")

### Define Metrics Function

In [ ]:
# Cell 10: Define Metrics Function
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    labels = labels.astype(np.int64) # Ensure labels are integers
    return accuracy_metric.compute(predictions=predictions, references=labels)

print("✅ compute_metrics function defined.")

### Initialize and Train the Model

In [ ]:
# Cell 11: Initialize and Train the Model

# Check if all necessary variables exist
if 'model' in locals() and model is not None and \
   'training_args' in locals() and \
   'tokenized_datasets' in locals() and tokenized_datasets is not None and \
   'tokenizer' in locals() and \
   'compute_metrics' in locals():

    trainer = Trainer( # Use standard trainer
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    print("--- Starting Model Training on Combined Data ---")
    try:
        trainer.train()
        print("✅ Combined model training complete!")
    except Exception as e:
        print(f"Error during training: {e}")
        # Ensure trainer object exists even if training fails, for potential debugging/saving
        if 'trainer' not in locals():
             trainer = None

else:
    print("Error: One or more required variables not defined. Cannot initialize Trainer.")
    trainer = None # Ensure trainer is None if setup fails

### Evaluate the Final Model

In [ ]:
# Cell 12: Evaluate the Final Model (Combined Data)

if 'trainer' in locals() and trainer is not None:
    print("\n--- Evaluating Combined Model on Test Set ---")
    try:
        eval_results_combined = trainer.evaluate()
        print("\n--- Test Set Performance ---")
        for key, value in eval_results_combined.items():
              print(f"{key}: {value:.4f}")

        # --- Generate Detailed Classification Report ---
        print("\n--- Generating Detailed Classification Report ---")
        predictions_output = trainer.predict(tokenized_datasets["test"])
        y_pred = np.argmax(predictions_output.predictions, axis=-1)
        y_true = np.array(tokenized_datasets["test"]["label"]) # Convert labels to NumPy array

        # Get the text names for our labels (ensure label2id is available)
        if 'label2id' in locals():
             target_names = sorted(label2id, key=label2id.get) # Get names in order of ID
        else:
             print("Warning: label2id not found, attempting to get labels from model config.")
             target_names = list(model.config.id2label.values())

        print("\n--- Classification Report (Combined Data) ---")
        print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

        # --- Confusion Matrix ---
        print("\n--- Confusion Matrix (Combined Data) ---")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10, 7))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=target_names, yticklabels=target_names)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix (Combined Data)')
        plt.show()

    except Exception as e:
        print(f"Error during evaluation: {e}")
else:
    print("Trainer object not found or training failed. Cannot evaluate.")

### Save the Final Model's Tokenizer

In [ ]:
# Cell 13: Save the Final Model's Tokenizer (Combined Data)

if 'trainer' in locals() and trainer is not None and 'tokenizer' in locals():
    final_combined_model_path = training_args.output_dir

    try:
        tokenizer.save_pretrained(final_combined_model_path)
        print(f"\n✅ Best combined model saved by Trainer to: {final_combined_model_path}")
        print(f"✅ Tokenizer saved to: {final_combined_model_path}")
    except Exception as e:
        print(f"Error saving tokenizer: {e}")
else:
    print("Trainer or tokenizer object not found. Cannot save.")

### Test the Model with Pipeline

In [ ]:
# Cell 14: Test the Model with Pipeline (Load from BEST Checkpoint)
import torch
from transformers import pipeline
import os

# --- Define project paths again for clarity ---
PROJECT_FOLDER = "/content/drive/My Drive/Harmful Language Detection/"
# --- Point to the specific BEST checkpoint directory ---
# Based on 3 epochs, the last checkpoint is likely best
BEST_CHECKPOINT_PATH = PROJECT_FOLDER + "high_end_intent_model_combined_v1/checkpoint-2040/" # ADJUST if needed

print(f"\n--- Attempting to load Model from Checkpoint: {BEST_CHECKPOINT_PATH} ---")

# Verify the checkpoint path exists and contains necessary files
if not os.path.isdir(BEST_CHECKPOINT_PATH):
    print(f"❌ Error: Checkpoint directory not found: {BEST_CHECKPOINT_PATH}")
    print("Please verify the correct checkpoint number (e.g., checkpoint-XXX based on training log steps).")
    classifier_pipeline_combined = None
else:
    # Check for config and weights
    config_ok = os.path.exists(os.path.join(BEST_CHECKPOINT_PATH, "config.json"))
    weights_ok = any(os.path.exists(os.path.join(BEST_CHECKPOINT_PATH, f)) for f in ["pytorch_model.bin", "model.safetensors"])

    if not config_ok:
        print(f"❌ Error: config.json not found inside {BEST_CHECKPOINT_PATH}")
    if not weights_ok:
        print(f"❌ Error: Model weights (pytorch_model.bin or model.safetensors) not found inside {BEST_CHECKPOINT_PATH}")

    if config_ok and weights_ok:
        print("✅ Found config.json and model weights file in checkpoint directory.")
        try:
            # Load the saved model using the 'pipeline' function pointing to the checkpoint
            classifier_pipeline_combined = pipeline(
                "text-classification",
                model=BEST_CHECKPOINT_PATH, # Load directly from the checkpoint
                tokenizer=BEST_CHECKPOINT_PATH, # Load tokenizer from checkpoint too
                device=0 if torch.cuda.is_available() else -1
            )
            print("✅ Combined model loaded successfully into pipeline from checkpoint.")

            # --- Test Sentences (same as before) ---
            test_sentences_combined = [
                "The quarterly earnings exceeded expectations.", # Expected: neutral
                "He admitted he took the money, but said he needed it.", # Expected: confession
                "Leave the package by the third dumpster behind the diner at 9 PM.", # Expected: planning_crime
                "If you don't approve this loan, you might find your car doesn't start tomorrow.", # Expected: threatening_action
                "They were talking about how they got away with the robbery last year.", # Expected: discussing_illegal_activity
                "This weather is terrible; I hope the flight isn't cancelled.", # Expected: neutral
                "Alright, I confess, I ate the last cookie.", # Expected: confession
                "You coordinate the lookout, I'll handle the entry.", # Expected: planning_crime
                "One wrong move and this whole place goes up in smoke.", # Expected: threatening_action
                "Did you hear about the big drug bust on the news?", # Expected: discussing_illegal_activity
                "Just send the invoice to accounting when you're done.", # Expected: neutral
                "We need to figure out how to get past the security checkpoint unnoticed.", # Expected: planning_crime
                "People who talk too much often have accidents.", # Expected: threatening_action
                "I should have known better than to trust him with the cash.", # Expected: confession
                "Apparently, the cops raided their hideout last night." # Expected: discussing_illegal_activity
            ]
            expected_labels_combined = [
                'neutral', 'confession', 'planning_crime', 'threatening_action', 'discussing_illegal_activity',
                'neutral', 'confession', 'planning_crime', 'threatening_action', 'discussing_illegal_activity',
                'neutral', 'planning_crime', 'threatening_action', 'confession', 'discussing_illegal_activity'
            ]

            # Get the predictions
            results_combined = classifier_pipeline_combined(test_sentences_combined)

            # Print the results
            print("\n--- Predictions on Test Sentences (Combined Model from Checkpoint) ---")
            correct_count = 0
            for sentence, result, expected in zip(test_sentences_combined, results_combined, expected_labels_combined):
                print(f"\nSentence:  '{sentence}'")
                print(f"Predicted: '{result['label']}' (Score: {result['score']:.4f})")
                print(f"Expected:  '{expected}'")
                if result['label'] == expected:
                    print("Result:    ✅ Correct")
                    correct_count += 1
                else:
                    print("Result:    ❌ Incorrect")
            print(f"\n--- Overall Test Sentence Accuracy: {correct_count / len(test_sentences_combined):.2%} ({correct_count}/{len(test_sentences_combined)}) ---")

        except Exception as e:
            print(f"Error loading pipeline or making predictions from checkpoint: {e}")
            classifier_pipeline_combined = None
    else:
         classifier_pipeline_combined = None # Ensure it's None if checks fail

# Optional check
if 'classifier_pipeline_combined' not in locals() or classifier_pipeline_combined is None:
    print("\nPipeline failed to load. Please check errors above.")

In [ ]:
# Sample dialogue corpus for testing criminal intent detection

dialogue_corpus = [
    {'speaker': 'Alex', 'dialogue': 'Hey guys, rough day at work. Glad it\'s finally over.'},
    {'speaker': 'Ben', 'dialogue': 'Tell me about it. My boss was breathing down my neck all afternoon.'},
    {'speaker': 'Chloe', 'dialogue': 'You guys want to grab a drink later? Maybe hit up that new bar downtown?'},
    {'speaker': 'Dave', 'dialogue': 'Can\'t tonight. Got... uh... some late-night errands to run.'},
    {'speaker': 'Alex', 'dialogue': 'Everything okay, Dave?'},
    {'speaker': 'Dave', 'dialogue': 'Yeah, yeah, just need to pick up a package. No big deal.'}, # Potentially ambiguous - planning?
    {'speaker': 'Ben', 'dialogue': 'Speaking of packages, remember that time we had to "deliver" that thing for Marco? That was intense.'}, # discussing_illegal_activity
    {'speaker': 'Chloe', 'dialogue': 'Ugh, don\'t remind me. I thought we were going to get caught.'}, # discussing_illegal_activity / neutral
    {'speaker': 'Alex', 'dialogue': 'Seriously, Ben, maybe keep those stories to yourself? Some things are better left unsaid.'}, # neutral / potentially trying to shut down discussion
    {'speaker': 'Dave', 'dialogue': 'Alex is right. Loose lips sink ships, you know?'}, # threatening_action (veiled)
    {'speaker': 'Ben', 'dialogue': 'Alright, alright, relax. I was just reminiscing.'},
    {'speaker': 'Chloe', 'dialogue': 'So, Alex, how did that presentation go?'},
    {'speaker': 'Alex', 'dialogue': 'Actually went really well, thanks for asking! The clients seemed impressed.'},
    {'speaker': 'Dave', 'dialogue': 'Good, good. Hey Ben, about that "errand"... meet me at the warehouse, Pier 12, midnight. Come alone.'}, # planning_crime
    {'speaker': 'Ben', 'dialogue': 'Midnight? Okay. Same plan as last time?'}, # planning_crime
    {'speaker': 'Dave', 'dialogue': 'Exactly. Disable the west camera first, then wait for my signal.'}, # planning_crime
    {'speaker': 'Chloe', 'dialogue': 'Warehouse? What are you guys talking about?'},
    {'speaker': 'Dave', 'dialogue': 'Just business, Chloe. Stuff you don\'t need to worry about. It\'s better if you don\'t ask questions.'}, # threatening_action (implied)
    {'speaker': 'Alex', 'dialogue': 'Sounds dodgy, Dave. Be careful, both of you.'},
    {'speaker': 'Ben', 'dialogue': 'Honestly? I\'m tired of this. That last job... I skimmed a little off the top. Took about five grand.'}, # confession
    {'speaker': 'Dave', 'dialogue': 'You WHAT?! Ben, you idiot! If Marco finds out, he\'ll kill you! And maybe us too!'}, # threatening_action (reporting potential threat) / discussing_illegal_activity
    {'speaker': 'Chloe', 'dialogue': 'Five grand? Ben, what were you thinking?!'},
    {'speaker': 'Ben', 'dialogue': 'I know, I know, it was stupid. I just needed the cash. I put it back mostly.'}, # confession
    {'speaker': 'Dave', 'dialogue': 'Mostly?! You better hope Marco never audits those books. And keep your mouth shut about this, understand? Or I\'ll shut it for you.'}, # threatening_action (direct)
    {'speaker': 'Alex', 'dialogue': 'Okay, this is getting out of hand. I\'m out of here. Stay safe... or whatever.'},
    {'speaker': 'Chloe', 'dialogue': 'Yeah, me too. Ben, seriously, fix this.'}
]

# You can now iterate through this list and feed the 'dialogue' into your pipeline
# Example:
for utterance in dialogue_corpus:
    speaker = utterance['speaker']
    text = utterance['dialogue']
    prediction = classifier_pipeline_combined(text)[0] # Get top prediction
    intent = prediction['label']
    score = prediction['score']
    print(f"[{speaker}]: '{text}' -> Predicted Intent: {intent} (Score: {score:.4f})")

print(f"✅ Sample dialogue corpus created with {len(dialogue_corpus)} utterances.")